In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append("..") 

In [ ]:
from processing import load_UCI_dataset,extract_all_features,extract_features_from_dataset, select_f_test, select_mrmr, select_reliefF, save_feature_set

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 625, STEP_SIZE= 312)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [01:20<00:00, 118.74it/s]


Skipped recordings: 6


100%|██████████| 1200/1200 [00:10<00:00, 116.03it/s]


Skipped recordings: 1


100%|██████████| 1200/1200 [00:10<00:00, 116.12it/s]


Skipped recordings: 0


In [ ]:
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (421829, 1, 625), y_train shape: (421829, 2)


## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

In [ ]:
test_window = X_train[0, 0, :]

features = extract_all_features(
    test_window,
    fs=125
)

print("Number of features:", len(features))

print(features)

Number of features: 57
{'ppg_mean': np.float64(2.154428934506354), 'ppg_median': np.float64(2.220918866080156), 'ppg_std': np.float64(0.5626222659163733), 'ppg_variance': np.float64(0.3165438141048743), 'ppg_iqr': np.float64(0.9921798631476049), 'ppg_skewness': np.float64(-0.27439930015112696), 'ppg_kurtosis': np.float64(-1.1298680721917524), 'ppg_zero_crossing_rate': np.float64(0.0), 'ppg_shannon_entropy': np.float64(5.493174555139509), 'ppg_energy_mean': np.float64(4.958107847943057), 'ppg_energy_variance': np.float64(5.543081302809521), 'ppg_energy_skewness': np.float64(0.03289597109053284), 'ppg_energy_kurtosis': np.float64(-1.3027001495614008), 'ppg_energy_iqr': np.float64(4.299446073639621), 'ppg_kte_mean': np.float64(0.00581370288614723), 'ppg_kte_variance': np.float64(0.00022242662925081073), 'ppg_kte_skewness': np.float64(0.12070837247417962), 'ppg_kte_kurtosis': np.float64(1.1143132201809731), 'ppg_kte_iqr': np.float64(0.014294396046358182), 'vpg_mean': np.float64(-0.00080076

In [ ]:
df_train = extract_features_from_dataset(
    X_train,
    y_train
)

df_val = extract_features_from_dataset(
    X_val,
    y_val
)

df_test = extract_features_from_dataset(
    X_test,
    y_test
)

  0%|          | 0/421829 [00:00<?, ?it/s]

100%|██████████| 53482/53482 [08:23<00:00, 106.31it/s]


In [ ]:
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Train: (421829, 59)
Validation: (53254, 59)
Test: (53482, 59)


## Feature Selection ##

We'll reduce the 57 features to 15 using 3 different methods

In [ ]:
feature_columns = [
    col for col in df_train.columns
    if col not in ["SBP", "DBP"]
]

X_train = df_train[feature_columns]
X_val = df_val[feature_columns]
X_test = df_test[feature_columns]

y_train_sbp = df_train["SBP"]
y_train_dbp = df_train["DBP"]

y_val_sbp = df_val["SBP"]
y_val_dbp = df_val["DBP"]

y_test_sbp = df_test["SBP"]
y_test_dbp = df_test["DBP"]

## F-Test ##

In [ ]:
# SBP
f_sbp_features, f_sbp_results = select_f_test(
    X_train,
    y_train_sbp,
    k=15
)
print("F-test selected features for SBP:")

for i, feature in enumerate(f_sbp_features, 1):
    print(i, feature)

F-test selected features for SBP:
1 vpg_skewness
2 apg_zero_crossing_rate
3 apg_skewness
4 ppg_iqr
5 ppg_energy_iqr
6 vpg_shannon_entropy
7 vpg_kurtosis
8 apg_shannon_entropy
9 ppg_variance
10 vpg_median
11 vpg_energy_skewness
12 ppg_energy_variance
13 ppg_std
14 vpg_iqr
15 ppg_kurtosis


In [ ]:
# DBP
f_dbp_features, f_dbp_results = select_f_test(
    X_train,
    y_train_dbp,
    k=15
)

print("\nF-test selected features for DBP:")

for i, feature in enumerate(f_dbp_features, 1):
    print(i, feature)


F-test selected features for DBP:
1 apg_iqr
2 vpg_median
3 ppg_kte_iqr
4 apg_std
5 apg_energy_iqr
6 vpg_kte_mean
7 apg_variance
8 apg_energy_mean
9 apg_kte_iqr
10 apg_kte_mean
11 vpg_kte_iqr
12 apg_median
13 ppg_kte_mean
14 vpg_variance
15 vpg_energy_mean


## mRMR ##

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_regression
from skrebate import ReliefF


# ============================================================
# mRMR FEATURE SELECTION
# ============================================================

def select_mrmr(
    X,
    y,
    k=15,
    sample_size=50000,
    random_state=42
):
    
    X = X.copy()
    y = np.asarray(y, dtype=np.float64)

    # --------------------------------------------------------
    # 1. Sample training data for feature selection
    # --------------------------------------------------------
    
    rng = np.random.RandomState(random_state)

    n_samples = min(sample_size, len(X))

    sample_indices = rng.choice(
        len(X),
        size=n_samples,
        replace=False
    )

    X_sample = X.iloc[sample_indices]
    y_sample = y[sample_indices]

    print(f"mRMR: using {n_samples} samples for feature selection")

    # Convert to numpy
    X_values = X_sample.values.astype(np.float64)
    y_values = y_sample.astype(np.float64)

    n_features = X_values.shape[1]

    # --------------------------------------------------------
    # 2. Relevance: MI(feature, target)
    # --------------------------------------------------------

    relevance = mutual_info_regression(
        X_values,
        y_values,
        random_state=random_state
    )

    # --------------------------------------------------------
    # 3. MI between every pair of features
    # --------------------------------------------------------

    redundancy = np.zeros(
        (n_features, n_features)
    )

    for i in range(n_features):

        for j in range(i + 1, n_features):

            mi_ij = mutual_info_regression(
                X_values[:, [i]],
                X_values[:, j],
                random_state=random_state
            )[0]

            redundancy[i, j] = mi_ij
            redundancy[j, i] = mi_ij

    # --------------------------------------------------------
    # 4. Greedy mRMR selection
    # --------------------------------------------------------

    selected = []

    # First feature = highest relevance
    first_feature = np.argmax(relevance)

    selected.append(first_feature)

    remaining = set(range(n_features))
    remaining.remove(first_feature)

    while len(selected) < k:

        best_feature = None
        best_score = -np.inf

        for candidate in remaining:

            # Average redundancy with
            # already selected features
            avg_redundancy = np.mean([
                redundancy[candidate, s]
                for s in selected
            ])

            # mRMR score
            score = (
                relevance[candidate]
                - avg_redundancy
            )

            if score > best_score:

                best_score = score
                best_feature = candidate

        selected.append(best_feature)
        remaining.remove(best_feature)

    # --------------------------------------------------------
    # 5. Get selected feature names
    # --------------------------------------------------------

    selected_features = [
        X.columns[i]
        for i in selected
    ]

    # --------------------------------------------------------
    # 6. Results table
    # --------------------------------------------------------

    results = pd.DataFrame({
        "feature": X.columns,
        "relevance_MI": relevance
    })

    results = results.sort_values(
        "relevance_MI",
        ascending=False
    ).reset_index(drop=True)

    return selected_features, results


# ============================================================
# RELIEFF FEATURE SELECTION
# ============================================================

def select_reliefF(
    X,
    y,
    k=15,
    n_neighbors=100,
    sample_size=50000,
    random_state=42
):

    X = X.copy()
    y = np.asarray(y)

    # --------------------------------------------------------
    # 1. Sample training data for feature selection
    # --------------------------------------------------------

    rng = np.random.RandomState(random_state)

    n_samples = min(sample_size, len(X))

    sample_indices = rng.choice(
        len(X),
        size=n_samples,
        replace=False
    )

    X_sample = X.iloc[sample_indices]
    y_sample = y[sample_indices]

    print(f"ReliefF: using {n_samples} samples for feature selection")

    # --------------------------------------------------------
    # 2. ReliefF model
    # --------------------------------------------------------

    model = ReliefF(
        n_neighbors=n_neighbors,
        n_features_to_select=k
    )

    model.fit(
        X_sample.values,
        y_sample
    )

    # --------------------------------------------------------
    # 3. Feature importance scores
    # --------------------------------------------------------

    scores = model.feature_importances_

    # Sort from highest to lowest
    indices = np.argsort(scores)[::-1][:k]

    selected_features = [
        X.columns[i]
        for i in indices
    ]

    # --------------------------------------------------------
    # 4. Results table
    # --------------------------------------------------------

    results = pd.DataFrame({
        "feature": X.columns,
        "ReliefF_score": scores
    })

    results = results.sort_values(
        "ReliefF_score",
        ascending=False
    ).reset_index(drop=True)

    return selected_features, results

In [ ]:
mrmr_sbp_features, mrmr_sbp_results = select_mrmr(
    X_train,
    y_train_sbp,
    k=15
)

print("mRMR selected features for SBP:")

for i, feature in enumerate(
    mrmr_sbp_features,
    1
):
    print(i, feature)

mRMR: using 50000 samples for feature selection
mRMR selected features for SBP:
1 apg_energy_variance
2 ppg_energy_kurtosis
3 ppg_zero_crossing_rate
4 vpg_skewness
5 apg_mean
6 ppg_energy_variance
7 apg_zero_crossing_rate
8 apg_kte_skewness
9 vpg_mean
10 apg_median
11 ppg_shannon_entropy
12 apg_skewness
13 apg_kte_variance
14 apg_shannon_entropy
15 vpg_median


In [ ]:
mrmr_dbp_features, mrmr_dbp_results = select_mrmr(
    X_train,
    y_train_dbp,
    k=15
)

print("\nmRMR selected features for DBP:")

for i, feature in enumerate(
    mrmr_dbp_features,
    1
):
    print(i, feature)

mRMR: using 50000 samples for feature selection

mRMR selected features for DBP:
1 apg_energy_variance
2 ppg_zero_crossing_rate
3 ppg_energy_skewness
4 apg_zero_crossing_rate
5 ppg_energy_variance
6 apg_mean
7 apg_kte_skewness
8 vpg_mean
9 apg_median
10 vpg_skewness
11 apg_kte_variance
12 ppg_shannon_entropy
13 apg_skewness
14 vpg_median
15 vpg_zero_crossing_rate


## ReliefF ##

In [ ]:
relief_sbp_features, relief_sbp_results = select_reliefF(
    X_train,
    y_train_sbp,
    k=15
)

print("ReliefF selected features for SBP:")

for i, feature in enumerate(
    relief_sbp_features,
    1
):
    print(i, feature)

ReliefF: using 50000 samples for feature selection
ReliefF selected features for SBP:
1 apg_median
2 vpg_median
3 apg_zero_crossing_rate
4 apg_skewness
5 vpg_skewness
6 vpg_iqr
7 vpg_zero_crossing_rate
8 vpg_shannon_entropy
9 ppg_shannon_entropy
10 apg_kte_iqr
11 ppg_skewness
12 vpg_kte_iqr
13 apg_iqr
14 apg_energy_iqr
15 ppg_energy_skewness


In [ ]:
relief_dbp_features, relief_dbp_results = select_reliefF(
    X_train,
    y_train_dbp,
    k=15
)

print("\nReliefF selected features for DBP:")

for i, feature in enumerate(
    relief_dbp_features,
    1
):
    print(i, feature)

ReliefF: using 50000 samples for feature selection

ReliefF selected features for DBP:
1 apg_zero_crossing_rate
2 apg_median
3 apg_skewness
4 vpg_median
5 vpg_skewness
6 vpg_iqr
7 vpg_shannon_entropy
8 vpg_zero_crossing_rate
9 ppg_shannon_entropy
10 ppg_skewness
11 apg_shannon_entropy
12 apg_kte_iqr
13 ppg_energy_skewness
14 ppg_kurtosis
15 apg_energy_iqr


In [ ]:
selection_summary = {

    "SBP_F_test": f_sbp_features,

    "SBP_mRMR": mrmr_sbp_features,

    "SBP_ReliefF": relief_sbp_features,

    "DBP_F_test": f_dbp_features,

    "DBP_mRMR": mrmr_dbp_features,

    "DBP_ReliefF": relief_dbp_features
}

In [ ]:
for name, features in selection_summary.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for i, feature in enumerate(features, 1):
        print(f"{i:2d}. {feature}")


SBP_F_test
 1. vpg_skewness
 2. apg_zero_crossing_rate
 3. apg_skewness
 4. ppg_iqr
 5. ppg_energy_iqr
 6. vpg_shannon_entropy
 7. vpg_kurtosis
 8. apg_shannon_entropy
 9. ppg_variance
10. vpg_median
11. vpg_energy_skewness
12. ppg_energy_variance
13. ppg_std
14. vpg_iqr
15. ppg_kurtosis

SBP_mRMR
 1. apg_energy_variance
 2. ppg_energy_kurtosis
 3. ppg_zero_crossing_rate
 4. vpg_skewness
 5. apg_mean
 6. ppg_energy_variance
 7. apg_zero_crossing_rate
 8. apg_kte_skewness
 9. vpg_mean
10. apg_median
11. ppg_shannon_entropy
12. apg_skewness
13. apg_kte_variance
14. apg_shannon_entropy
15. vpg_median

SBP_ReliefF
 1. apg_median
 2. vpg_median
 3. apg_zero_crossing_rate
 4. apg_skewness
 5. vpg_skewness
 6. vpg_iqr
 7. vpg_zero_crossing_rate
 8. vpg_shannon_entropy
 9. ppg_shannon_entropy
10. apg_kte_iqr
11. ppg_skewness
12. vpg_kte_iqr
13. apg_iqr
14. apg_energy_iqr
15. ppg_energy_skewness

DBP_F_test
 1. apg_iqr
 2. vpg_median
 3. ppg_kte_iqr
 4. apg_std
 5. apg_energy_iqr
 6. vpg_kte_m

## Saving ##

In [ ]:
save_feature_set(
    name="SBP_FTEST",
    selected_features=f_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_MRMR",
    selected_features=mrmr_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_RELIEFF",
    selected_features=relief_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: SBP_FTEST
Number of selected features: 15
Train shape: (421829, 16)
Val shape:   (53254, 16)
Test shape:  (53482, 16)

Saved:
data/selected_feature_sets/SBP_FTEST_train.csv
data/selected_feature_sets/SBP_FTEST_val.csv
data/selected_feature_sets/SBP_FTEST_test.csv
data/selected_feature_sets/SBP_FTEST_preprocessing.pkl

Processing: SBP_MRMR
Number of selected features: 15
Train shape: (421829, 16)
Val shape:   (53254, 16)
Test shape:  (53482, 16)

Saved:
data/selected_feature_sets/SBP_MRMR_train.csv
data/selected_feature_sets/SBP_MRMR_val.csv
data/selected_feature_sets/SBP_MRMR_test.csv
data/selected_feature_sets/SBP_MRMR_preprocessing.pkl

Processing: SBP_RELIEFF
Number of selected features: 15
Train shape: (421829, 16)
Val shape:   (53254, 16)
Test shape:  (53482, 16)

Saved:
data/selected_feature_sets/SBP_RELIEFF_train.csv
data/selected_feature_sets/SBP_RELIEFF_val.csv
data/selected_feature_sets/SBP_RELIEFF_test.csv
data/selected_feature_sets/SBP_RELIEFF_preprocessing.pkl


In [ ]:
save_feature_set(
    name="DBP_FTEST",
    selected_features=f_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_MRMR",
    selected_features=mrmr_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_RELIEFF",
    selected_features=relief_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: DBP_FTEST
Number of selected features: 15
Train shape: (421829, 16)
Val shape:   (53254, 16)
Test shape:  (53482, 16)

Saved:
data/selected_feature_sets/DBP_FTEST_train.csv
data/selected_feature_sets/DBP_FTEST_val.csv
data/selected_feature_sets/DBP_FTEST_test.csv
data/selected_feature_sets/DBP_FTEST_preprocessing.pkl

Processing: DBP_MRMR
Number of selected features: 15
Train shape: (421829, 16)
Val shape:   (53254, 16)
Test shape:  (53482, 16)

Saved:
data/selected_feature_sets/DBP_MRMR_train.csv
data/selected_feature_sets/DBP_MRMR_val.csv
data/selected_feature_sets/DBP_MRMR_test.csv
data/selected_feature_sets/DBP_MRMR_preprocessing.pkl

Processing: DBP_RELIEFF
Number of selected features: 15
Train shape: (421829, 16)
Val shape:   (53254, 16)
Test shape:  (53482, 16)

Saved:
data/selected_feature_sets/DBP_RELIEFF_train.csv
data/selected_feature_sets/DBP_RELIEFF_val.csv
data/selected_feature_sets/DBP_RELIEFF_test.csv
data/selected_feature_sets/DBP_RELIEFF_preprocessing.pkl
